In [1]:
# == Imports ==

import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import scipy.signal as signal
from datasets import load_dataset
from IPython.display import Audio, display
from dotenv import load_dotenv
import jiwer
import platform
import time
import warnings
from pprint import pprint
import gradio as gr

# Filter out the benign duplicate logits processor warnings from transformers
warnings.filterwarnings("ignore", message=".*logits processor of type.*")

load_dotenv()

# Set figsize global params
plt.rcParams['figure.figsize'] = (12,5)

---

# Load dataset

In [2]:
# == Load dataset in stream mode ==

print(f"Loading LibriSpeech dataset stream...")
dataset = load_dataset("openslr/librispeech_asr", "clean", split="test", streaming=True)

Loading LibriSpeech dataset stream...


Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

In [3]:
# == Grab first sample ==
iterator = iter(dataset)
sample = next(iterator)
# sample2 = next(iterator)
# sample2 = next(iterator)
audio_array = sample["audio"]["array"]
sample_rate = sample["audio"]["sampling_rate"]
ground_truth_text = sample['text']

print(f"Sample Rate: {sample_rate} Hz")
print(f"Ground truth text: {ground_truth_text}")

Sample Rate: 16000 Hz
Ground truth text: CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS


In [ ]:
# == Plot the clean waveform ==
librosa.display.waveshow(audio_array, sr=sample_rate)
plt.title('Clean Speech Waveform - Sample 1')
plt.show()

In [ ]:
# == Listen to the audio ==
Audio(audio_array, rate=sample_rate)

---

# Metrics and Utility Functions

In [4]:
# == fct: normalize_text() ==
cleanup_pipeline = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.Strip(),
    # jiwer.ReduceToListOfListOfWords()
])

# def normalize_text(text):
#     return jiwer.RemovePunctuation()(text.lower().strip())


In [5]:
# == fct: analyze_asr_audio_health ==

def analyze_asr_audio_health(y, frame_length=2048, hop_length=512):
    """
    Calculates audio metrics and flags structural issues that degrade ASR
    Assumes y is a 1D float32 numpy array
    Returns a dictionary of metrics and ASR flags
    """
    eps = 1e-10

    # ----------------------------------------
    # 1. Base energy and metrics
    # ----------------------------------------
    # Global RMS
    rms_linear_global = np.sqrt(np.mean(y**2))
    rms_dbfs_global = 20 * np.log10(np.maximum(rms_linear_global, eps))

    # Frame-based RMS
    rms_frames = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
    rms_dbfs_frames = 20 * np.log10(np.maximum(rms_frames, eps))

    # Linear and DBFS Peaks
    abs_y = np.abs(y)
    peak_max_linear = np.max(abs_y)
    peak_max_dbfs = 20 * np.log10(np.maximum(peak_max_linear, eps))

    # ----------------------------------------
    # 2. ASR Specific Diagnostic Metrics
    # ----------------------------------------
    # DC Offset: Should be very close to 0.0
    dc_offset = np.mean(y)

    # Crest Factor: Peak to RMS Ratio in dB
    crest_factor_db = peak_max_dbfs - rms_dbfs_global
    #
    # Zero-Crossing Rate: High ZCR + Low RMS = Pure Noise
    zcr_frames = librosa.feature.zero_crossing_rate(y=y, frame_length=frame_length, hop_length=hop_length)[0]
    zcr_mean = np.mean(zcr_frames)

    # ----------------------------------------
    # 3. Automated ASR Health Flags
    # ----------------------------------------
    # Clip Detection: Values Tracking exactly at 1.0 or -1.0
    is_clipping = np.sum(abs_y >=.99) > (len(y) * 0.001)  # More than 0.1% of sampels

    # Too Quiet Detection: ASR models struggle below -40 dBFS RMS
    is_too_quiet = rms_dbfs_global < -40.0

    # Pure Noise/Silence Detection: High zero crossing but no real energy
    # is_pure_noise = (zcr_mean > 0.15) and (rms_dbfs_global < -30.0)

    zcr_min = np.min(zcr_frames)    # Vowels always drop ZCR low, noise dow not
    zcr_std = np.std(zcr_frames)    # Speech has high ZCR variance; static has low variance
    # If the minimum frame ZCR is high, it means the signal never became harmonic (no vowels)
    is_pure_noise = (zcr_min > 0.12) and (rms_dbfs_global < -25.0)

    round_factor = 4
    return {
        "metrics": {
            "rms_lin_global": round(float(rms_linear_global), round_factor),
            "rms_dbfs_global": round(float(rms_dbfs_global), round_factor),
            "rms_dbfs_min_frame": round(float(np.min(rms_dbfs_frames)), round_factor),
            "rms_dbfs_max_frame": round(float(np.max(rms_dbfs_frames)), round_factor),
            "linear_range": [round(float(np.min(y)), round_factor), round(float(np.max(y)), round_factor)],
            "peak_dbfs": round(float(peak_max_dbfs), round_factor),
            "dc_offset": round(float(dc_offset), round_factor),
            "crest_factor_db": round(float(crest_factor_db), round_factor),
            "min_zcr": round(float(zcr_min), round_factor),
            "std_zcr": round(float(zcr_std), round_factor),
            "mean_zcr": round(float(zcr_mean), round_factor)
        },
        "asr_flags": {
            "action_required": bool(is_clipping or is_too_quiet or is_pure_noise or (abs(dc_offset) > .01)),
            "reasons": {
                "digital_clipping_detected": bool(is_clipping),
                "signal_too_faint": bool(is_too_quiet),
                "likely_static_or_hum": bool(is_pure_noise),
                "dc_offset_warning": bool(abs(dc_offset) > 0.01)
            }
        }
    }

In [6]:
# == tone generator ==

def generate_sine_tone(frequency=440.0, duration=2.0, sr=16_000, amplitude=1.0):
    """
    Generate a pure sine tone array perfectly compatible with librosa for testing.
    """
    # Create the time axis array
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)

    # Generate the sine wave and scale by amplitude (1.0 is full scale / 0 dBFS peak)
    y = amplitude * np.sin(2 * np.pi * frequency * t)

    # Ensure it uses float32
    return y.astype(np.float32), sr

---

# Audio Modification/Corruption Functions

## White Gaussian Noise

In [8]:
# == fct: add_white_noise ==

def add_white_noise(signal, target_snr_db):
    """
    Inject additive white Gaussian noise scaled to a precise target SNR (dB)
    """
    # Calculate RMS of clean signal
    rms_signal = np.sqrt(np.mean(np.square(signal)))

    # Calculate the required RMS of noise based on target SNR in dB
    rms_noise = rms_signal / (10**(target_snr_db/20))

    # Generate noise and scale it
    noise_raw = np.random.randn(len(signal))
    rms_noise_raw = np.sqrt(np.mean(np.square(noise_raw)))

    # Scale the noise to match the target noise RMS
    noise_scaled = noise_raw * (rms_noise / rms_noise_raw)

    signal_with_noise = signal + noise_scaled

    if np.max(np.abs(signal_with_noise))>1.0:
        print("Warning: Mixed audio exceeds 0 dBFS. Normalizing to prevent clipping.")
        signal_with_noise = signal_with_noise / np.max(np.abs(signal_with_noise))

    return signal_with_noise


### White Noise tests

In [ ]:
test_signal_0dB = add_white_noise(audio_array, 0)
pprint(analyze_asr_audio_health(test_signal_0dB))
librosa.display.waveshow(test_signal_0dB, sr=sample_rate, alpha=0.5, label='noisy')
librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
plt.title(f"Audio Waveform Overlay ({0} dB SNR White Noise)")
plt.legend()
plt.show()

Audio(test_signal_0dB, rate=sample_rate)

In [ ]:
test_signal_3dB = add_white_noise(audio_array, 3)
pprint(analyze_asr_audio_health(test_signal_3dB))
librosa.display.waveshow(test_signal_3dB, sr=sample_rate, alpha=0.5, label='noisy')
librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
plt.title(f"Audio Waveform Overlay ({3} dB SNR White Noise)")
plt.legend()
plt.show()
Audio(test_signal_3dB, rate=sample_rate)


In [ ]:
test_signal_6dB = add_white_noise(audio_array, 6)
pprint(analyze_asr_audio_health(test_signal_6dB))
librosa.display.waveshow(test_signal_6dB, sr=sample_rate, alpha=0.5, label='noisy')
librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
plt.title(f"Audio Waveform Overlay ({6} dB SNR White Noise)")
plt.legend()
plt.show()
Audio(test_signal_6dB, rate=sample_rate)


In [ ]:
test_signal_9dB = add_white_noise(audio_array, 9)
pprint(analyze_asr_audio_health(test_signal_9dB))
librosa.display.waveshow(test_signal_9dB, sr=sample_rate, alpha=0.5, label='noisy')
librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
plt.title(f"Audio Waveform Overlay ({9} dB SNR White Noise)")
plt.legend()
plt.show()
Audio(test_signal_9dB, rate=sample_rate)

In [ ]:
test_signal_12dB = add_white_noise(audio_array, 12)
pprint(analyze_asr_audio_health(test_signal_12dB))
librosa.display.waveshow(test_signal_12dB, sr=sample_rate, alpha=0.5, label='noisy')
librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
plt.title(f"Audio Waveform Overlay ({12} dB SNR White Noise)")
plt.legend()
plt.show()
Audio(test_signal_12dB, rate=sample_rate)

In [ ]:
test_signal_20dB = add_white_noise(audio_array, 20)
pprint(analyze_asr_audio_health(test_signal_20dB))
librosa.display.waveshow(test_signal_20dB, sr=sample_rate, alpha=0.5, label='noisy')
librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
plt.title(f"Audio Waveform Overlay ({20} dB SNR White Noise)")
plt.legend()
plt.show()
Audio(test_signal_20dB, rate=sample_rate)

In [ ]:
test_signal_neg3dB = add_white_noise(audio_array, -3)
pprint(analyze_asr_audio_health(test_signal_neg3dB))
librosa.display.waveshow(test_signal_neg3dB, sr=sample_rate, alpha=0.5, label='noisy')
librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
plt.title(f"Audio Waveform Overlay ({-3} dB SNR White Noise)")
plt.legend()
plt.show()
Audio(test_signal_neg3dB, rate=sample_rate)


In [ ]:
test_signal_neg6dB = add_white_noise(audio_array, -6)
pprint(analyze_asr_audio_health(test_signal_neg6dB))
librosa.display.waveshow(test_signal_neg6dB, sr=sample_rate, alpha=0.5, label='noisy')
librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
plt.title(f"Audio Waveform Overlay ({-6} dB SNR White Noise)")
plt.legend()
plt.show()
Audio(test_signal_neg6dB, rate=sample_rate)

## Pink and Brown Noise

In [ ]:
# def add_colored_noise(signal, target_snr_db, color=pink)

## Reverberation

In [9]:
# == fct: add_synthetic_reverberation() ==

def add_synthetic_reverberation(signal, sr=16_000, rt60=0.5, mix=0.4):
    """
    Synthesizes an exponential decay Impulse Response (IR) to simulate room acoustics,
    then convolve it with the clean speech signal

    rt6-: Time (seconds) for the reverb tail to drop by 60dB.
    mix: dry/wet mix balance between direct signal and reverberation
    """
    ir_len = int(rt60 * sr)
    t = np.linspace(0, rt60, ir_len)

    # Generate white noise and apply clean mathematical exponential devay envelope
    # -6.91 guarantees the amplitude drops exactly 60 dB over the course of rt60
    decay_envelope = np.exp(-6.91 * t / rt60)
    ir = np.random.randn(ir_len) * decay_envelope

    # Normalize the IR energy fader to prevent massive volume scaling jumps
    # In signal processing, we use L2 normalization to keep the total amount of enery = 1
    ir = ir / np.sqrt(np.sum(ir**2))

    # Convolve the signal with our custom synthetic room IR
    reverb_signal = np.convolve(signal, ir, mode='full')[:len(signal)]

    # Apply standard Wet/Dry mixing balance
    mixed_signal = ( 1-mix ) * signal + mix * reverb_signal

    # Safety rail guard to prevent hard clipping
    if np.max(np.abs(mixed_signal)) > 1.0:
        mixed_signal = mixed_signal / np.max(np.abs(mixed_signal))

    return mixed_signal

In [10]:
# == fct: add_real_reverberation() ==

def extract_ir_from_mit_dataset(dataset, id, target_sr=16_000):
    """
    Retreive the IR audio array fromthe MIT Impulse Respons survey dataset based on its id
    """
    sample = dataset.filter(lambda sample: sample["id"]==id)
    decoder = sample["audio"][0]
    audio_tensor = decoder.get_all_samples()
    y_native = audio_tensor.data.numpy()
    sr_native = sample["audio"][0].metadata.sample_rate

    y = librosa.resample(y_native, orig_sr=sr_native, target_sr=target_sr)

    return y, target_sr

def extract_irs_from_mit_dataset_by_location(dataset, location, target_sr=16_000):
    """
    Retreive the IR audio array fromthe MIT Impulse Respons survey dataset based on its id
    """
    examples = dataset.filter(lambda sample: sample["location"]==location)

    results = []

    for example in examples:
        id = example["id"]
        detail = example["detail"]
        decoder = example["audio"]
        audio_tensor = decoder.get_all_samples()
        y_native = audio_tensor.data.numpy()
        sr_native = example["audio"].metadata.sample_rate
        duration = example["audio"].metadata.duration_seconds

        y = librosa.resample(y_native, orig_sr=sr_native, target_sr=target_sr)

        results.append(
            {'id': id,
             'location': location,
             'detail': detail,
             'duration': duration,
             'audio': y,
             'sr': target_sr
             })

    return results



def add_real_reverberation(signal, ir, sr=16_000, mix=0.4):
    """
    Convolves a clean audio signal with a recorder physical Impulse Response
    """

    # Force mono on both signal and IR
    # if len(signal.shape)> 1:
    #     signal = np.mean(signal, axis=1)
    # if len(ir.shape) > 1:
    #     ir = np.mean(ir, axis=0)

    # Flatten ir array if necessary
    if ir.shape[0]==1:
        ir = ir.flatten()


    # Normalize the IR energy so it doesn't artificially boost the overall volume
    # using L-infinity normalization to ensure that the high-amplitude transient of the impulse response
    # does not cause L2 normalization to overly attenuate the decay tail.
    # ir_scaled = ir / np.max(np.abs(ir))
    # ir_scaled = ir / np.sqrt(np.sum(ir**2))
    #
    rms = np.sqrt(np.mean(ir**2))
    target_rms = 0.05
    ir_scaled = ir * (target_rms / rms)
    ir_scaled = np.clip(ir_scaled, -1.0, 1.0)

    # Convolve the signal with the real room IR
    reverb_signal = np.convolve(signal, ir_scaled, mode='full')[:len(signal)]

    # Apply the wer/dry balance
    mixed_signal = ( 1 - mix ) * signal + mix * reverb_signal


    # Safety prevention of clipping
    if np.max(np.abs(mixed_signal)) > 1.0:
        mixed_signal = mixed_signal / np.max(np.abs(mixed_signal))

    # Convert float32 [-1.0, 1.0] to int16 [-32786, 32767]
    mixed_signal_int16 = (mixed_signal * 32767).astype(np.int16)

    return mixed_signal_int16

### Inspection of different normlization effect on IR RMS and dynamic

In [ ]:
test_ir = irs[1]['audio']
librosa.display.waveshow(audio_array, color='blue', alpha=0.5)
librosa.display.waveshow(test_ir, color='red', alpha=0.5)

In [ ]:
l_inf = test_ir / np.max(np.abs(test_ir))
librosa.display.waveshow(l_inf, color='red', alpha=0.5)

In [ ]:
l2 = test_ir / np.sqrt(np.sum(test_ir**2))
librosa.display.waveshow(l_inf, color='red', alpha=0.5)

In [ ]:
rms = np.sqrt(np.mean(test_ir**2))
target_rms = 0.03
ir_scaled = test_ir * (target_rms / rms)
ir_scaled = np.clip(ir_scaled, -1.0, 1.0)
librosa.display.waveshow(ir_scaled, color='red', alpha=0.5)

In [ ]:
# == real room IR reverberation ==
ir_ds = load_dataset("benjamin-paine/mit-impulse-response-survey", split="train")

location_id2label = ir_ds.features["location"].int2str
location_str2id = ir_ds.features["location"].str2int
ir_ds_locations = ir_ds.features['location'].names
pprint(ir_ds_locations)

In [ ]:
irs = extract_irs_from_mit_dataset_by_location(ir_ds, location_str2id("Stairwell"))
print("".join([f"id: {e['id']} \ndetail: {e['detail']} \nduration in secs: {e['duration']} \n---------\n\n" for e in irs]))

In [ ]:
gr.close_all()

with gr.Blocks() as demo:
    n = len(irs)
    gr.Markdown(f"# {n} different IRs applied to audio")
    with gr.Column():
        for ir in irs:
            audio = add_real_reverberation(audio_array, ir["audio"])
            gr_audio = (16_000, audio)
            output = gr.Audio(gr_audio, label=f"{location_id2label(ir['location'])}: {ir['detail']} | {round(ir['duration'],2)} secs")

demo.launch(prevent_thread_lock=True)

In [ ]:
gr.close_all()

## Dropout

In [16]:
## == Dropout function ==

def add_smart_audio_dropouts(signal, sr=16_000, num_dropouts=3, dropout_duration_ms=100, threshold_below_global_db=15):
    """
    Simulates network packet loss or a bad connection by zeroing out
    chunks of audio at a random interval timestamps
    """
    degraded_signal = signal.copy()
    dropout_len = int((dropout_duration_ms / 1000.0) * sr)

    # Ensure dropout zone is bounded within track length
    if len(signal) <= dropout_len:
        return degraded_signal

    possible_starts = len(signal) - dropout_len

    dropouts_applied = 0
    attempts = 0
    max_attempts = 50
    eps = 1e-10
    rms_linear_global = np.sqrt(np.mean(signal**2))
    rms_dbfs_global = 20 * np.log10(np.maximum(rms_linear_global, eps))

    adaptive_silence_threshold = rms_dbfs_global - threshold_below_global_db

    while dropouts_applied < num_dropouts  and attempts < max_attempts:
        # Pick a random start index
        start = np.random.randint(0, possible_starts)

        # Extract the targeted chunk of audio
        target = signal[start: start + dropout_len]

        # Claculate RMS energy of this specific chunk and convert to dBFS
        target_rms_linear = np.sqrt(np.mean(target**2))
        target_rms_dbfs = 20 * np.log10(np.maximum(target_rms_linear, eps))

        if target_rms_dbfs > adaptive_silence_threshold:
            # Accepted chunk as speech
            degraded_signal[start : start + dropout_len] = 0.0
            dropouts_applied+=1

        attempts+=1

    return degraded_signal


In [ ]:
test = add_smart_audio_dropouts(audio_array)
Audio(test, rate=16_000)

---

# Transcriber Engine

In [ ]:
# == setup classifier pieline ==
def load_transcriber_engine(IS_APPLE_SILICON=False):
    """
    Automatically detect if Apple Silicon and tries to load mlx_whisper if available
    If not available, falls back to HuggingFace pipeline and setup device accordingly ("cuda", "mps", "cpu")

    Needs GLOBAL VAR: IS_APPLE_SILICON  to be set True or False
    :return:
        if MLX is used
    """

    if IS_APPLE_SILICON:
        try:
            import mlx_whisper
            print("Apple Silicon Detected. Initializing MLX.")

            def mlx_transcriber(audio_array):
                # Using MLX community pre-converted weights of the turbo model
                result = mlx_whisper.transcribe(audio_array, path_or_hf_repo="mlx-community/whisper-large-v3-turbo")
                return {"text": result["text"]}
            return mlx_transcriber

        except ImportError:
            print(f"Apple Silicon detected but 'mlx_whisper' not installed. Falling back to PyTorch MPS.")

    # 2 Standard Production/Linux Fallback
    from transformers import pipeline, AutoTokenizer, GenerationConfig
    import torch

    print("Standard architecture detected. Initializing Hugging Face pipeline.")
    device = "cuda" if torch.cuda.is_available() else "mps" #("mps" if IS_APPLE_SILICON else "cpu")
    dtype = torch.float32 if device=="mps" else torch.float16

    model_id = "openai/whisper-large-v3-turbo"
    tokenizer = AutoTokenizer.from_pretrained(model_id, clean_up_tokenization_spaces=False)

    whisper_config = GenerationConfig.from_pretrained(model_id)
    whisper_config.task = "transcribe"
    whisper_config.language = "en"
    # whisper_config.num_beams = 1

    whisper_pipe = pipeline(
        "automatic-speech-recognition",
        model=model_id,
        tokenizer=tokenizer,
        dtype=dtype,
        device=device
    )

    # Enclose pipeline in a wrapper function to pass the manual config along
    def hf_transcriber(audio_array):
        result = whisper_pipe(audio_array, generation_config=whisper_config)
        return result

    return hf_transcriber

---
# TESTS

In [ ]:
# == Hardware detection ==
IS_APPLE_SILICON = platform.system() == "Darwin" and platform.processor() == "arm"
# IS_APPLE_SILICON= False
transcriber = load_transcriber_engine(IS_APPLE_SILICON)

In [ ]:
# == fct: run_benchmark_test() ==

def run_benchmark_test(n_samples=10):   # no noise
    iterator = iter(dataset)
    target_snr = 3  # signal-to-noise ratio in dB to be added to the samples

    running_avg_wer_clean = 0
    running_avg_wer_noisy = 0

    total_words_ref = 0
    clean_total_hits = 0
    clean_total_subs = 0
    clean_total_dels = 0
    clean_total_ins = 0
    noisy_total_hits = 0
    noisy_total_subs = 0
    noisy_total_dels = 0
    noisy_total_ins = 0

    results = []

    for i in range(n_samples):
        sample = next(iterator)
        audio_array = sample["audio"]["array"]
        sample_rate = sample["audio"]["sampling_rate"]
        noisy_audio_array = add_white_noise(audio_array, target_snr)
        clean_ref = cleanup_pipeline(sample["text"])
        clean_hyp = cleanup_pipeline(transcriber(audio_array)['text'])
        noisy_hyp = cleanup_pipeline(transcriber(noisy_audio_array)['text'])

        metrics_clean = jiwer.process_words(clean_ref, clean_hyp)
        metrics_noisy = jiwer.process_words(clean_ref, noisy_hyp)

        c_hits = metrics_clean.hits
        c_subs = metrics_clean.substitutions
        c_dels = metrics_clean.deletions
        c_ins = metrics_clean.insertions
        n_hits = metrics_noisy.hits
        n_subs = metrics_noisy.substitutions
        n_dels = metrics_noisy.deletions
        n_ins = metrics_noisy.insertions
        n_words = c_hits + c_subs + c_dels
        c_wer = metrics_clean.wer
        n_wer = metrics_noisy.wer

        # print(f"\n=== SAMPLE {i+1} ===")
        # print(f"Clean ref: \n{clean_ref}\n")
        # print(f"Clean hyp: \n{clean_hyp}\n")
        # print(f"Noisy hyp: \n{noisy_hyp}\n")
        # print(f"WER clean: {c_wer}")
        # print(f"WER noisy: {n_wer}")

        # # update WER running avg
        # k = i+1
        # running_avg_wer_clean = running_avg_wer_clean*(k-1)/k + wer_clean/k
        # running_avg_wer_noisy = running_avg_wer_noisy*(k-1)/k + wer_noisy/k

        total_words_ref += n_words
        clean_total_hits += c_hits
        clean_total_subs += c_subs
        clean_total_dels += c_dels
        clean_total_ins += c_ins
        noisy_total_hits += n_hits
        noisy_total_subs += n_subs
        noisy_total_dels += n_dels
        noisy_total_ins += n_ins

        # Store results and references into list for further analysis
        sample_analysis = analyze_asr_audio_health(audio_array)
        noisy_sample_analysis = analyze_asr_audio_health(noisy_audio_array)

        results.append({"idx": i,
                        "c_wer": c_wer,
                        "c_hits": c_hits,
                        "c_subs": c_subs,
                        "c_dels": c_dels,
                        "c_ins": c_ins,
                        "n_wer": n_wer,
                        "n_hits": n_hits,
                        "n_subs": n_subs,
                        "n_dels": n_dels,
                        "n_ins": n_ins,
                        "n_words": n_words,
                        "c_analysis": sample_analysis,
                        "n_analysis": noisy_sample_analysis,
                        "c_ref": clean_ref,
                        "c_hyp": clean_hyp,
                        "n_hyp": noisy_hyp})

    # calculate total metrics
    c_wer_total = (clean_total_subs+clean_total_dels+clean_total_ins) / total_words_ref
    n_wer_total = (noisy_total_subs+noisy_total_dels+noisy_total_ins) / total_words_ref
    print(f"Final metrics:")
    print(f"WER clean:  {c_wer_total}")
    print(f"WER noisy:  {n_wer_total}")
    print(f"\n--- Breakdown ---")
    print(f"Total words:      {total_words_ref}")
    print(f"Clean:")
    print(f"\tHits (Correct): {clean_total_hits}")
    print(f"\tSubs:           {clean_total_subs}")
    print(f"\tDels:           {clean_total_dels}")
    print(f"\tIns:            {clean_total_ins}")
    print(f"Noisy:")
    print(f"\tHits (Correct): {noisy_total_hits}")
    print(f"\tSubs:           {noisy_total_subs}")
    print(f"\tDels:           {noisy_total_dels}")
    print(f"\tIns:            {noisy_total_ins}")

    return results
#


In [ ]:
bench_test_results = run_benchmark_test(n_samples=50)

In [ ]:
test_df = pd.json_normalize(bench_test_results)
cols = [col.split(".") for col in test_df.columns]

exclude = {"metrics", "asr_flags", "reasons"}

cleaned_cols = [
    ".".join(
        item.replace("c_analysis", "c_anly")
        .replace("n_analysis", "n_anly")
        .replace("digital_clipping_detected", "clipping_detected")
        for item in sublist
        if item not in exclude
    )
    for sublist in cols
]

test_df.columns = cleaned_cols
test_df.info()
# pprint(cleaned_cols)

In [ ]:
# == Some data engineering ==
test_df["delta_wer"] = test_df["n_wer"] - test_df["c_wer"]
test_df["delta_crest_factor"] = test_df["n_anly.crest_factor_db"] - test_df["c_anly.crest_factor_db"]
test_df["delta_mean_zcr"] = test_df["n_anly.mean_zcr"] - test_df["c_anly.mean_zcr"]
test_df["delta_std_zcr"] = test_df["n_anly.std_zcr"] - test_df["c_anly.std_zcr"]
test_df["delta_min_zcr"] = test_df["n_anly.min_zcr"] - test_df["c_anly.min_zcr"]
test_df["delta_rms"] = test_df["n_anly.rms_dbfs_global"] - test_df["c_anly.rms_dbfs_global"]
test_df.sort_values(by=["c_wer", "n_wer"], ascending=False, inplace=True)

test_df

In [ ]:
# == generate column lists per type
num_cols = test_df.select_dtypes(include=['number']).columns.tolist()
num_cols.remove("idx")
cat_cols = test_df.select_dtypes(include=['str', 'object', 'category']).columns.tolist()

print(f"numerical columns: \n\t{num_cols}")
print(f"categorical columns: \n\t{cat_cols}")

In [ ]:
# == Look at cross-correlation between numerical variables ==
cols_to_correlate = [
    'delta_wer',
    'n_anly.crest_factor_db',
    'n_anly.mean_zcr',
    'n_anly.rms_dbfs_global',
    'c_anly.mean_zcr',
    'c_anly.rms_dbfs_global',
    'delta_crest_factor',
    'delta_mean_zcr',
    'delta_std_zcr',
    'delta_min_zcr',
    'delta_rms']

corr_df = test_df[cols_to_correlate].dropna()
corr_matrix = corr_df.corr(method='spearman')

sns.heatmap(corr_matrix,
            annot=True,
            cmap='coolwarm',
            fmt='.2f',
            vmin=-1, vmax=1)
plt.title("Correlation: Audio Metrics vs WER Degradation")
plt.show()


In [ ]:
# ==  scatter plot of delta_wer and delta_zcr ==
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Noisy ZCR vs WER
sns.scatterplot(x='n_anly.mean_zcr', y='delta_wer', data=test_df, ax=axes[0], alpha=0.6)
axes[0].set_title('Noisy ZCR vs WER Degradation')
axes[0].set_xlabel("Noisy Audio ZCR")
axes[0].set_ylabel("WER delta")

# Plot 2: Original RMS vs WER
sns.scatterplot(x='c_anly.rms_dbfs_global', y='delta_wer', data=test_df, ax=axes[1], alpha=0.6)
axes[1].set_title('Original Clean RMS vs WER Degradation')
axes[1].set_xlabel('Clean RMS (dBFS)')
axes[1].set_ylabel('WER Delta')

plt.tight_layout()
plt.show()

In [ ]:
# Plot Noist ZCR vs Delta WER with colored dots for RMS dBFS
sns.scatterplot(
    x='n_anly.mean_zcr',
    y='delta_wer',
    hue='delta_min_zcr',
    palette='viridis',
    data=test_df,
    alpha=0.8,
    s=60
)

plt.title('Noisy ZCR vs WER Degradation (Colored by delta_min_zcr)')
plt.xlabel('Noisy Audio ZCR')
plt.ylabel('WER Delta (Degradation)')

# Move the legend outside the plot
plt.legend(title='Delta min ZCR', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

---

## Framework Speed Comparison (MLX vs MPS)

In [ ]:
# IS_APPLE_SILICON = platform.system() == "Darwin" and platform.processor() == "arm"
# IS_APPLE_SILICON= False
# transcriber = load_transcriber_engine()

def run_framework_speed_comparison(n_samples=10):
    # 1.load both engines
    mlx_transcriber = load_transcriber_engine(IS_APPLE_SILICON=True)
    mps_transcriber = load_transcriber_engine(IS_APPLE_SILICON=False)

    # 2.Pre-load all samples to prevent biasing results from network latencies
    iterator = iter(dataset)
    samples = [next(iterator) for _ in range(n_samples + 1)]
    warmup_audio = samples[0]["audio"]["array"]
    test_samples = samples[1:]

    print(f"Warming up engines...")
    _ = mlx_transcriber(warmup_audio)
    _ = mps_transcriber(warmup_audio)

    # Pre-load ground truth text refs
    refs = []
    for s in test_samples:
        refs.append(cleanup_pipeline(s["text"]))

    # 3. Time engine A (MLX)
    print(f"Profiling Native MLX...")
    mlx_hyps = []
    mlx_start = time.perf_counter()
    for s in test_samples:
        mlx_hyps.append(cleanup_pipeline(mlx_transcriber(s["audio"]["array"])["text"]))
    mlx_time = time.perf_counter() - mlx_start

    # 4. Time engine 4 (PyTorch MPS)
    print(f"Profiling PyTorch MPS pipeline...")
    mps_hyps =  []
    mps_start = time.perf_counter()
    for s in test_samples:
        mps_hyps.append(cleanup_pipeline(mps_transcriber(s["audio"]["array"])["text"]))
    mps_time = time.perf_counter() - mps_start

    # 5. Compute Metrics
    mlx_wer = jiwer.wer(refs, mlx_hyps)
    mps_wer = jiwer.wer(refs, mps_hyps)

    print(f"\nMLX Total Time   : {mlx_time:.2f}s | WER: {mlx_wer*100:.2f}%")
    print(f"PyTorch MPS Time : {mps_time:.2f}s | WER: {mps_wer*100:.2f}%")
    print(f"MLX Speed Factor : {mps_time / mlx_time:.2f}x faster")

    return

run_framework_speed_comparison()


<img src="./images/huge_mps_wer_result.png" alt="Aternative Text" width="600">
<img src="./images/huge_mps_wer.png" alt="Aternative Text" width="600">

In [ ]:
noise_test = np.random.randn(1000000)
plt.hist(noise_test, bins=np.arange(-5, 5, .1))

plt.show()

In [ ]:
plt.plot(noise_test)
plt.show()

In [ ]:
target_noise_rms = .5
noise_test_ratiod = (noise_test / np.std(noise_test)) * target_noise_rms
plt.plot(noise_test_ratiod)
plt.show()

In [ ]:
def add_white_noise_test(signal, target_snr_db):
    """
    Inject additive white Gaussian noise scaled to a precise target SNR (dB)
    """
    # Calculate RMS of clean signal
    rms_signal = np.sqrt(np.mean(np.square(signal)))
    print(f"raw signal metrics:     min:{np.min(signal):.2f} | max:{np.max(signal):.2f} | mean:{np.mean(signal):.2f} | std:{np.std(signal):.2f}")
    print(f"rms signal: {rms_signal}")

    # Calculated DBFS


    # Calculate the required RMS of noise based on target SNR in dB
    rms_noise = rms_signal / (10**(target_snr_db/20))
    print(f"rms noise required: {rms_noise}")

    # Generate noise and scale it
    noise_raw = np.random.randn(len(signal))
    rms_noise_raw = np.sqrt(np.mean(np.square(noise_raw)))
    print(f"noise_raw metrics:     min:{np.min(noise_raw):.2f} | max:{np.max(noise_raw):.2f} | mean:{np.mean(noise_raw):.2f} | std:{np.std(noise_raw):.2f}")
    print(f"rms_noise_raw:         {rms_noise_raw}")

    # Scale the noise to match the target noise RMS
    noise_scaled = noise_raw * (rms_noise / rms_noise_raw)
    print(f"noise_scaled metrics: min:{np.min(noise_scaled):.2f} | max:{np.max(noise_scaled):.2f} | mean:{np.mean(noise_scaled):.2f} | std:{np.std(noise_scaled):.2f}")

    signal_with_noise = signal + noise_scaled
    print(f"signal_with_noise metrics: min:{np.min(signal_with_noise):.2f} | max:{np.max(signal_with_noise):.2f} | mean:{np.mean(signal_with_noise):.2f} | std:{np.std(signal_with_noise):.2f}")

    if np.max(np.abs(signal_with_noise))>1.0:
        print("Warning: Mixed audio exceeds 0 dBFS. Normalizing to prevent clipping.")
        signal_with_noise = signal_with_noise / np.max(np.abs(signal_with_noise))

    return signal_with_noise

In [ ]:
librosa.display.waveshow(audio_array, sr=sample_rate)
Audio(audio_array, rate=16_000)

In [ ]:
audio_array_w_noise_snr_6db = add_white_noise_test(audio_array, 6)

In [ ]:
0.029361743479967117 / .014715731143951416

In [ ]:
plt.plot(audio_array, color='blue', alpha=0.5, label="raw signal")
plt.plot(np.abs(audio_array), color='red', alpha=0.3, label="abs. signal")
plt.axhline(y=np.mean(np.abs(audio_array)), color="red", linestyle="--", label="mean")
plt.axhline(y=np.mean(np.square(audio_array)), color="blue", linestyle="--", label="mean of squared")
plt.axhline(y=np.sqrt(np.mean(np.square(audio_array))), color="blue", linestyle="--", label="RMS")
plt.legend()
plt.show()

In [ ]:
import soundfile as sf

In [ ]:
sf.write("sample_1.wav", audio_array, sample_rate, format="WAV", subtype="PCM_16")

In [ ]:
tone_1khz_array, tone_1khz_sr = librosa.load("../data/raw/1khz_tone_0DBFS.wav")

In [ ]:
1.0 / np.sqrt(2.0)

In [ ]:
analyze_asr_audio_health(audio_array)

In [ ]:
analyze_asr_audio_health(tone_1khz_array)

In [ ]:
# test funciton with 1khz pure tone at 0 dBFS

analyze_asr_audio_health(generate_sine_tone(frequency=1000.0)[0])